In [0]:
%pip install sqlalchemy psycopg2-binary
dbutils.library.restartPython()

In [0]:
from urllib.parse import quote_plus
from sqlalchemy import create_engine

POSTGRES_HOST = "ep-aged-sound-a8rv8vt7-pooler.eastus2.azure.neon.tech"
POSTGRES_PORT = "5432"
POSTGRES_DB = "Ingest-Tables"
POSTGRES_USER = "neondb_owner"
POSTGRES_PASSWORD = "npg_c4BMOi2wuhmG"  # Cambia esta contraseña primero

POSTGRES_URL = (
    f"postgresql+psycopg2://{POSTGRES_USER}:{quote_plus(POSTGRES_PASSWORD)}"
    f"@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}?sslmode=require"
)

engine = create_engine(POSTGRES_URL)

In [0]:
from datetime import datetime, timedelta, timezone
# Tipo de carga
dbutils.widgets.text("ultima_fecha_carga", "2026-03-21", "Ultima_Fecha_Carga")
dbutils.widgets.text("tabla", "ventas", "Tabla")
dbutils.widgets.text("tipo_carga", "F", "Tipo_Carga")
dbutils.widgets.text("columna_fecha", "fecha_venta", "Columna_Fecha")
dbutils.widgets.text("tipo_tabla", "transaccional", "Tipo de Tabla")

# Leer valores
tabla = dbutils.widgets.get("tabla")
tipo_carga = dbutils.widgets.get("tipo_carga")
start_date = dbutils.widgets.get("ultima_fecha_carga")
column_date = dbutils.widgets.get("columna_fecha")
tipo_tabla = dbutils.widgets.get("tipo_tabla")

# Fecha de hoy menos 1
# 1. Creamos la zona horaria específica de Colombia (UTC-5)
tz_colombia = timezone(timedelta(hours=-5))

# 2. Obtenemos la fecha y hora exacta anclada a Colombia
ahora_colombia = datetime.now(tz_colombia)

# 3. Le restamos un día para obtener TU "ayer" real
ayer_colombia = ahora_colombia - timedelta(days=1)

# 4. Extraemos solo la fecha en formato ISO (ej. "2026-03-22")
end_date = ayer_colombia.date().isoformat()

print(f"Tabla seleccionada: {tabla}")
print(f"start_date: {start_date}")
print(f"end_date: {end_date}")
print(f"column_date: {column_date}")
print(f"tipo_carga: {tipo_carga}")

In [0]:
import pandas as pd
if tipo_carga == "F":
    query = f"SELECT * FROM {tabla}"
    print(f"Query: {query}")
elif tipo_carga == "I":
    query = f"SELECT * FROM {tabla} WHERE {column_date} > '{start_date}' AND {column_date} <= '{end_date}'"
    print(f"Query: {query}")
else:
    raise ValueError("Tipo de carga no válido")

df = pd.read_sql(query, engine)
spark_df = spark.createDataFrame(df)

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
schema = "bronze"
full_table_name = f"workspace.{schema}.{tabla}"
print(full_table_name)


In [0]:
from datetime import datetime, timedelta

if tipo_carga == "F":
    # FULL:
    # Sobrescribe toda la tabla
    (
        spark_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")   # quítalo si no quieres tocar esquema
        .saveAsTable(full_table_name)
    )
    print(f"Carga FULL completada en {full_table_name}")

elif tipo_carga == "I":
    # INCREMENTAL:
    
    delete_sql = f"""
    DELETE FROM {full_table_name}
    WHERE {column_date} > TIMESTAMP('{start_date}')
      AND {column_date} <=  TIMESTAMP('{end_date}')
    """
    spark.sql(delete_sql)

    (
        spark_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(full_table_name)
    )
    print(f"Carga INCREMENTAL completada en {full_table_name}")

In [0]:
def upsert_ultima_fecha_carga(df, columna_fecha, nombre_tabla, tabla_control):
    """
    Calcula la fecha máxima de una columna del DataFrame
    y hace upsert en la tabla de control:
    - si nombre_tabla ya existe -> UPDATE
    - si no existe -> INSERT

    Parámetros:
    - df: DataFrame origen
    - columna_fecha: nombre de la columna de fecha
    - nombre_tabla: nombre lógico de la tabla origen, por ejemplo 'ventas'
    - tabla_control: nombre completo de la tabla control, por ejemplo:
                     'mi_catalogo.mi_esquema.control_cargas'
    """

    # 1) Obtener la fecha máxima
    max_fecha = (
        df.select(F.max(F.col(columna_fecha).cast("date")).alias("max_fecha"))
          .first()["max_fecha"]
    )

    if max_fecha is None:
        raise ValueError(f"No se encontró una fecha válida en la columna {columna_fecha}")

    # 2) Crear DataFrame fuente de una sola fila
    df_upsert = spark.createDataFrame(
        [(nombre_tabla, max_fecha)],
        ["nombre_tabla", "ultima_fecha_carga"]
    )

    # 3) Referenciar la tabla Delta de control
    delta_target = DeltaTable.forName(spark, tabla_control)

    # 4) MERGE: update si existe, insert si no existe
    (
        delta_target.alias("t")
        .merge(
            df_upsert.alias("s"),
            "t.nombre_tabla = s.nombre_tabla"
        )
        .whenMatchedUpdate(set={
            "nombre_tabla": "s.nombre_tabla",
            "ultima_fecha_carga": "s.ultima_fecha_carga"
        })
        .whenNotMatchedInsert(values={
            "nombre_tabla": "s.nombre_tabla",
            "ultima_fecha_carga": "s.ultima_fecha_carga"
        })
        .execute()
    )

    return max_fecha
    
if tipo_tabla == "transaccional":
    max_fecha = upsert_ultima_fecha_carga(
        df=spark_df,
        columna_fecha=column_date,
        nombre_tabla=tabla,
        tabla_control="workspace.bronze.metadata"
    )

In [0]:
# Asegúrate de que la variable "tabla" tenga el nombre correcto
if tipo_tabla == "transaccional":
    spark.sql(f"""
    UPDATE workspace.bronze.metadata
    SET tipo_carga = 'I'
    WHERE nombre_tabla = '{tabla}' 
""")
    
    print(f"Metadata actualizada a Incremental (I) para la tabla: {tabla}")